## Project Description 
The Goal of this Project is to compute the global minimum variance (GMV) portfolio for the top $p=400$ stocks of the S&P500 over the course of $n = 26$ weeks. To do this we will need to compute the weekly excess returns matrix $M$ (p x n), then subtract the mean from each row to obtain the de-meaned excess returns matrix. 

The next step is to compute the sample covariance matrix $S = \frac{YY^T}{n}$ (p x p). Traditionally for a dataset where $n \leq p$ the covariance matrix is always invertible due to it having full rank. In this case where $p >> n$ the covariance metrix is only capable of having a rank of at most $26$. To deal with this we can create a single factor market model for the covariance matrix using the formula $\Sigma = (\lambda^2 - \ell^2)hh^T + (n/p)\ell^2 I$ where $n=26$ is the number of observations, $p=400$ is the number of assets, $\lambda^2$ is the leading/largest eigenvalue of $S$, $h$ is its correspoding unit eigenvector, $I$ is the identity matrix, and $\ell^2= \frac{tr(S)- \lambda^2}{n-1}$ is the average of the non-zero eigenvalues that are less then $\lambda^2$. 

Add awnsers to questions here of invertibility and leading eigenval etc.. 

Now that the covariance matrix is invertible we can proceed to calculate the GMV portfolio holding vector along with its variance, standard deviation, beta, and expected returns. we will then compare this again the individual variances of the stock within the portfolio


## Import Necessary Libraries

In [1]:
import numpy as np
import requests, pandas as pd
import yfinance as yf
from io import StringIO

## Web Scrape for Stock Tickers
The first challenge this project faces is gathering the tickers associated with all the stocks in the S&P500. this can be done using the official Wikipedia API to request structured data directly from the server. This involves setting up the request by setting the enpoint for the API, then parsing through the page that includes the list of S&P500 companies and requesting the test of the article. Then to bypass the restrictions, we set a user agent to give the script an identity. The request then fetches the content as a JSON object and extracts the table via the pandas and StringIO libraries. The final step is to extract the tickers from the table and store them as a list. 

Attachehed here is a reference link to the video for the code - https://www.youtube.com/watch?v=9gsn_9lh6uk

In [2]:
# Define wikipedia API endpoint 
API = 'https://en.wikipedia.org/w/api.php'

# Parameters to request the page parsed as JSON
params = {
    'action': 'parse',
    'page': 'List_of_S&P_500_companies', 
    'prop': 'text', 
    'format': 'json'
}

# Identity header to prevent the API from blocking the request
headers = {'User-Agent': 'sp500-ticker-tutorial/1.0'}
r = requests.get(API, params = params, headers = headers)
html = r.json()['parse']['text']['*']

# Read the HTML table into a DataFrame and extract the symbol column as a list
df = pd.read_html(StringIO(html))[0]
tickers = df['Symbol'].astype(str).tolist()

## Data Manipulation and Organization

In [3]:
# Remove unusable tickers for yfinance api 
for bad in ["BRK.B", "BF.B"]:
    if bad in tickers:
        tickers.remove(bad)

# add usable tickers
tickers = tickers + ["BRK-B","BF-B"]

# Define Date Range
start_date = "2025-05-26"
end_date = "2025-11-28"

# Download raw data for all tickers 
data = yf.download(tickers, start=start_date, end=end_date, interval="1wk")
closing_prices = data['Close']

# initialize storage for dictionary
results = [] 

# loop through each ticker 
for ticker in tickers:
    try:
        # grab the most recent price and the number of outstanding shares
        last_price = closing_prices[ticker].dropna().iloc[-1] 
        stock = yf.Ticker(ticker)
        shares = stock.info.get('sharesOutstanding')

        if shares != None:
            # Compute market cap for individual tickers
            m_cap = last_price * shares
            # append ticker and associated market cap data to dictionary
            results.append({'Ticker': ticker, 'MarketCap': m_cap})
            
    except Exception: 
        continue 

# convert to dataframe for easier use
dataframe = pd.DataFrame(results)
# sort from largest to smallest and grab top 400 elements
top_400 = dataframe.sort_values(by='MarketCap', ascending=False).head(400)

# store top 400 tickers in list
top_400_tickers = top_400["Ticker"].tolist() 
# Download data again but this time for top 400 tickers
top_400_data = yf.download(top_400_tickers, start= start_date, end = end_date, interval = "1wk")
top_400_close = top_400_data["Close"]

# Check for missing values in data 
missing_counts = top_400_close.isna().sum()
if missing_counts == 0:
    print("The Data contains no missing values")
else: 
    print("The Data contains missing values")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  503 of 503 completed
[*********************100%***********************]  400 of 400 completed


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## Global Minimum Variance Portfolio Computations

In [ ]:
returns = top_400_close.pct_change().dropna() # the second dropna is to ensure proper shape
returns_matrix = returns.T.to_numpy()

risk_free_rate = 0.0409 # 10-yr treasury bonds 
weekly_RFR = risk_free_rate/52 

# Excess Returns Formula
excess_returns_matrix = returns_matrix - weekly_RFR 

# axis=1 takes mean of each row and creates a column vector of each stocks mean 
# keepdims=True enable matrix subtraction by copying each collumn to match the matrix being worked on  
excess_returns_mean_vector = excess_returns_matrix.mean(axis=1, keepdims= True)

# Creating the p x n de-meaned excess returns matrix Y
Y = excess_returns_matrix - excess_returns_mean_vector

# calculate sample covariance matrix S
# Since n (26) < p (400), S is singular (non-invertible)
S = (Y @ Y.T) / 26 

# compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(S)

# find leading eigenvalue (Market Factor)
lambda_sq = np.real(eigenvalues[-1])
h = np.real(eigenvectors[:,-1]).reshape(-1,1)

# subtract the leading eigenvalue from the total variance (Trace)
# Distribute the remainder across the other n-1 degrees of freedom
trace_S = np.real(np.trace(S))
ell_sq = (trace_S - lambda_sq) / (25)

# Assemble Single-Factor Model Covariance Matrix
term1 = (lambda_sq - ell_sq) * (h @ h.T)
term2 = (26/400) * ell_sq * np.eye(400)
Sigma = np.real(term1 + term2)

eiganvalues_Sigma = np.linalg.eigvals(Sigma)
index_sigma = np.argmax(eiganvalues_Sigma)

# e is vector of ones used to enforce constraints thast weights sum to 1
e = np.ones(400) # characteristic for GMV portfolio
Sigma_inverse = np.linalg.inv(Sigma) 

# Compute Global Minimum Variance (GMV) weights (h_C)
holdings_C = (Sigma_inverse @ e) / (e.T @ Sigma_inverse @ e)

# Calculate the variance 
Variance_C = (holdings_C.T @ Sigma @ holdings_C)
annual_Variance_C = Variance_C * 52 

# Compute Standard Deviation 
std_pfolo_C = np.sqrt(Variance_C)
annualized_std_pfolio_C = std_pfolo_C * np.sqrt(52)

# Compute expected excess returns
expected_excess_returns_C = holdings_C.T @ excess_returns_mean_vector
annualized_expected_excess_returns_C = expected_excess_returns_C * 52

# Calculate Beta WRT Pfolio C
Beta_C = (Sigma @ holdings_C) / Variance_C

#Compute Variance for indivdual stocks 
# store the diagonal elements in array 
individual_variance = np.diag(Sigma)
annual_individual_variance = individual_variance * 52
mean_variance = annual_individual_variance.mean()

print("Risk Free Rate", risk_free_rate)
print("Returns Matrix Dimensions:", returns_matrix.shape)
print("Cov. Matrix Dimensions:", S.shape)
print("Eigenvector Dimensions:", h.shape)
print("Leading Eigenvalue :", lambda_sq)
print("Leading Eiganvalue for Factor Model Covariance", np.real(eiganvalues_Sigma[index_sigma]))
print("Matrix Dimension:", Sigma.shape)
print("Total Holdings Vector Sum:", sum(holdings_C))
print("Pfolio C Variance:", annual_Variance_C)
print("Portfolio C Standard Deviation:", annualized_std_pfolio_C)
print("Expected Excess Returns for Portfolio C:", annualized_expected_excess_returns_C)
print("Pfolio C Beta Length: ", len(Beta_C))
print("Pfolio C Beta Sum: ",  sum(Beta_C))
print("Mean of individual stock variances:", mean_variance)
print("Mean of individual stock Std Dev:", np.sqrt(mean_variance))